1.

In [2]:
import pandas as pd
import numpy as np


data = {
    "CustomerID": [
        "C101", "C102", "C103", "C104", "C105",
        "C105", None, "C107", "C108", "C109"
    ],

    "CustomerName": [
        "Raj Kumar", "Priya S", "Arun V", "Kavya R", "Rahul P",
        "Rahul P", "Sneha M", "Deepak N", "Meena K", "John D"
    ],

    "Gender": [
        "Male", "Female", "Male", "Female", "Male",
        "Male", "Female", "Male", "Male", "Male"
    ],

    "Age": [
        28, 35, 150, 0, 42,
        42, 30, None, 27, 31
    ],

    "ProductCategory": [
        "Electronics", "Fashion", "Electronics", "Grocery", "Fashion",
        "Fashion", "Electronics", "Grocery", "Fashion", "Electronics"
    ],

    "Quantity": [
        2, 3, 1, -2, 5,
        5, 2, 3, 1, 4
    ],

    "UnitPrice": [
        "15000 INR", "1200 INR", "300 USD", "800 INR", "2500 INR",
        "2500 INR", "450 USD", "600 INR", "1800 INR", "250 EUR"
    ],

    "PurchaseDate": [
        "10-01-2025", "11-01-2025", "12-01-2025", "15-01-2025",
        "18-01-2025", "18-01-2025", "05-02-2025", "07-02-2025",
        "2025-13-01", "15-02-2025"
    ],

    "City": [
        "Chennai", "chennai", "CHENNAI", "Coimbatore", "Madurai",
        "Madurai", "Trichy", "Tiruchirappalli", "Salem", "Chennai"
    ]
}

df = pd.DataFrame(data)

print("\n============================================================")
print("ORIGINAL E-COMMERCE DATA")
print("============================================================")

print(df)

print("\n\n============================================================")
print("QUESTION 1: IDENTIFY AND CLASSIFY DATA QUALITY ISSUES")
print("============================================================")


# Missing Customer IDs
print("\n1. Missing Customer IDs - COMPLETENESS ISSUE")
print(df[df["CustomerID"].isnull()])


# Duplicate transactions
print("\n2. Duplicate Transactions - UNIQUENESS ISSUE")
print(df[df.duplicated(keep=False)])


# Negative quantities
print("\n3. Negative Quantities - VALIDITY ISSUE")
print(df[df["Quantity"] < 0])


# Different currencies
print("\n4. Different Currencies - CONSISTENCY ISSUE")
print(df["UnitPrice"])


# Invalid ages
print("\n5. Invalid Ages - VALIDITY ISSUE")
print(df[(df["Age"] <= 0) | (df["Age"] > 100)])


# Missing age
print("\n6. Missing Age - COMPLETENESS ISSUE")
print(df[df["Age"].isnull()])


# Inconsistent cities
print("\n7. Inconsistent City Names - CONSISTENCY ISSUE")
print(df["City"].unique())


# Invalid dates
print("\n8. Invalid Purchase Date - VALIDITY ISSUE")

test_dates = pd.to_datetime(
    df["PurchaseDate"],
    dayfirst=True,
    errors="coerce"
)

print(df[test_dates.isnull()])


print("\nSUMMARY OF DATA QUALITY ISSUES")
print("--------------------------------")
print("Missing Customer ID       -> Completeness")
print("Duplicate Transactions    -> Uniqueness")
print("Negative Quantity         -> Validity")
print("Different Currencies      -> Consistency")
print("Invalid Ages              -> Validity")
print("Missing Age               -> Completeness")
print("Inconsistent City Names   -> Consistency")
print("Invalid Purchase Date     -> Validity")


print("\n\n============================================================")
print("QUESTION 2: CLEANING STRATEGY FOR EACH ISSUE")
print("============================================================")

print("""
1. Missing Customer ID
   -> Check the original source.
   -> If the ID cannot be recovered, assign UNKNOWN_001.

2. Duplicate Transactions
   -> Identify duplicate rows.
   -> Remove exact duplicate transactions.

3. Negative Quantity
   -> Check whether it represents a return.
   -> If it is an error, remove or correct the transaction.

4. Different Currencies
   -> Convert USD and EUR into INR.
   -> Use one standard currency for revenue calculation.

5. Invalid Age
   -> Ages 0 and 150 are invalid.
   -> Replace invalid values with the median valid age.

6. Missing Age
   -> Replace missing age using the median valid age.

7. Inconsistent City Names
   -> Convert city names to a common format.
   -> Example: Chennai, chennai, CHENNAI -> Chennai.

8. Invalid Purchase Date
   -> Convert dates into standard date format.
   -> Invalid dates become NaT and should be verified from the source.
""")


print("\n\n============================================================")
print("QUESTION 3: DATA STANDARDIZATION AND TRANSFORMATION")
print("============================================================")


df["CustomerID"] = df["CustomerID"].fillna("UNKNOWN_001")

df = df.drop_duplicates()


df.loc[
    (df["Age"] <= 0) | (df["Age"] > 100),
    "Age"
] = np.nan

median_age = df["Age"].median()

df["Age"] = df["Age"].fillna(median_age)

df = df[df["Quantity"] >= 0]


df["City"] = df["City"].str.strip().str.title()

df["City"] = df["City"].replace({
    "Trichy": "Tiruchirappalli"
})

USD_TO_INR = 85
EUR_TO_INR = 95


def convert_to_inr(price):

    price = str(price)

    if "INR" in price:
        value = float(
            price.replace("INR", "").strip()
        )
        return value

    elif "USD" in price:
        value = float(
            price.replace("USD", "").strip()
        )
        return value * USD_TO_INR

    elif "EUR" in price:
        value = float(
            price.replace("EUR", "").strip()
        )
        return value * EUR_TO_INR

    return np.nan


df["UnitPrice_INR"] = df["UnitPrice"].apply(convert_to_inr)
df["PurchaseDate"] = pd.to_datetime(
    df["PurchaseDate"],
    dayfirst=True,
    errors="coerce"
)

df["Revenue"] = (
    df["Quantity"] *
    df["UnitPrice_INR"]
)


print("\nCLEANED AND STANDARDIZED DATA")
print("--------------------------------")

print(
    df[
        [
            "CustomerID",
            "CustomerName",
            "Gender",
            "Age",
            "ProductCategory",
            "Quantity",
            "UnitPrice_INR",
            "PurchaseDate",
            "City",
            "Revenue"
        ]
    ]
)


print("\n\n============================================================")
print("QUESTION 4: EFFECT OF CLEANING ON TOTAL REVENUE")
print("============================================================")

# Extract numeric price from original data
def extract_price(price):

    price = str(price)

    return float(
        price.split()[0]
    )


raw_df = pd.DataFrame(data)

raw_df["NumericPrice"] = raw_df["UnitPrice"].apply(
    extract_price
)

raw_df["RawRevenue"] = (
    raw_df["Quantity"] *
    raw_df["NumericPrice"]
)

raw_revenue = raw_df["RawRevenue"].sum()


# ------------------------------------------------------------
# Calculate CLEAN revenue
# ------------------------------------------------------------

clean_revenue = df["Revenue"].sum()


# ------------------------------------------------------------
# Display comparison
# ------------------------------------------------------------

print("\nRAW REVENUE")
print("--------------------------------")
print("Raw Revenue =", raw_revenue)


print("\nCLEANED REVENUE")
print("--------------------------------")
print("Cleaned Revenue =", clean_revenue)


print("\nREVENUE DIFFERENCE")
print("--------------------------------")
print("Difference =", clean_revenue - raw_revenue)


print("\nREVENUE ANALYSIS")
print("--------------------------------")

print("""
The raw revenue is unreliable because:

1. Duplicate transactions increase revenue.
2. Negative quantities produce incorrect revenue.
3. USD and EUR are not directly comparable with INR.
4. Invalid records can affect revenue calculations.
5. Cleaning produces a more reliable revenue value.
""")


# ============================================================
# QUESTION 5
# BUSINESS RISKS IF DATA IS NOT CLEANED
# ============================================================

print("\n\n============================================================")
print("QUESTION 5: BUSINESS RISKS IF DATA IS NOT CLEANED")
print("============================================================")


print("""
1. INCORRECT REVENUE REPORTING
   Duplicate transactions can overstate total revenue.

2. WRONG CUSTOMER SEGMENTATION
   Missing and invalid customer information can result in
   incorrect customer segments.

3. INCORRECT PRODUCT PERFORMANCE
   Different currencies can make product revenue comparisons
   inaccurate.

4. POOR INVENTORY DECISIONS
   Incorrect sales data can result in wrong inventory planning.

5. WRONG MARKETING DECISIONS
   Incorrect customer and city information can cause marketing
   campaigns to target the wrong customers.

6. INCORRECT SALES FORECASTING
   Invalid dates, quantities and prices can reduce the accuracy
   of future sales forecasts.

7. FINANCIAL RISK
   Incorrect revenue values can affect financial reports,
   budgets and business performance measurements.

8. MANAGEMENT DECISION RISK
   Management may make incorrect decisions because the analysis
   is based on unreliable data.
""")


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n\n============================================================")
print("FINAL SUMMARY")
print("============================================================")

print("Original number of records :", len(raw_df))
print("Cleaned number of records  :", len(df))
print("Raw Revenue                :", raw_revenue, "INR")
print("Cleaned Revenue            :", clean_revenue, "INR")

print("\nTop Customer by Revenue:")

top_customer = (
    df.groupby("CustomerID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

print(top_customer.head(1))


print("\nTop Product Category by Revenue:")

top_product = (
    df.groupby("ProductCategory")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

print(top_product.head(1))


# ============================================================
# SAVE CLEANED DATA
# ============================================================

df.to_csv(
    "cleaned_ecommerce_transactions.csv",
    index=False
)

print("\nCleaned dataset saved successfully!")
print("File: cleaned_ecommerce_transactions.csv")


ORIGINAL E-COMMERCE DATA
  CustomerID CustomerName  Gender    Age ProductCategory  Quantity  UnitPrice  \
0       C101    Raj Kumar    Male   28.0     Electronics         2  15000 INR   
1       C102      Priya S  Female   35.0         Fashion         3   1200 INR   
2       C103       Arun V    Male  150.0     Electronics         1    300 USD   
3       C104      Kavya R  Female    0.0         Grocery        -2    800 INR   
4       C105      Rahul P    Male   42.0         Fashion         5   2500 INR   
5       C105      Rahul P    Male   42.0         Fashion         5   2500 INR   
6        NaN      Sneha M  Female   30.0     Electronics         2    450 USD   
7       C107     Deepak N    Male    NaN         Grocery         3    600 INR   
8       C108      Meena K    Male   27.0         Fashion         1   1800 INR   
9       C109       John D    Male   31.0     Electronics         4    250 EUR   

  PurchaseDate             City  
0   10-01-2025          Chennai  
1   11-01-2025

2.


In [3]:
import pandas as pd
import numpy as np

# Create DataFrame
data = {
    "PatientID": ["P1001","P1002","P1003",None,"P1005","P1005","P1006","P1007","P1008","P1009"],
    "Name": ["Ravi Kumar","Priya S","Arun V","Kavya R","Rahul P","Rahul P","Sneha M","Deepak N","Meena K","John D"],
    "Age": [45,38,125,29,51,51,34,None,26,40],
    "Gender": ["M","F","M","F","M","M","F","M","F","M"],
    "AdmissionDate": ["10-01-2025","12-01-2025","13-01-2025","15-01-2025","18-01-2025","18-01-2025","15-01-2025","01-02-2025","10-02-2025","12-02-2025"],
    "DischargeDate": ["15-01-2025","11-01-2025","20-01-2025","17-01-2025","25-01-2025","25-01-2025","18-01-2025","05-02-2025","15-02-2025","18-02-2025"],
    "Diagnosis": ["Diabetes","Hypertension","diabetes","Diabtes","Asthma","Asthma","Hypertension","Asthma","HYPERTENSION","Diabetes"],
    "TreatmentCost": [25000,18000,22000,15000,None,None,12000,10000,13500,20000],
    "DoctorID": ["D101","D102","D101","D103","D104","D104","D102","D104","D102","D101"]
}

df = pd.DataFrame(data)

print("\n" + "="*60)
print("QUESTION 1: DETECT INCONSISTENCIES AND ANOMALIES")
print("="*60)

print("\nMissing Patient IDs:")
print(df[df["PatientID"].isnull()])

print("\nDuplicate Records:")
print(df[df.duplicated(keep=False)])

print("\nDischarge Before Admission:")
ad = pd.to_datetime(df["AdmissionDate"], dayfirst=True)
dd = pd.to_datetime(df["DischargeDate"], dayfirst=True)
print(df[dd < ad])

print("\nMissing Treatment Cost:")
print(df[df["TreatmentCost"].isnull()])

print("\nInconsistent Diagnosis:")
print(df["Diagnosis"].unique())

print("\nUnrealistic Ages:")
print(df[df["Age"] > 120])

print("\nMissing Ages:")
print(df[df["Age"].isnull()])


print("\n" + "="*60)
print("QUESTION 2: RULES FOR MISSING AND INVALID VALUES")
print("="*60)

print("""
1. Missing PatientID -> Assign a temporary ID or retrieve from source.
2. Duplicate records -> Remove duplicate records.
3. Invalid dates -> Verify and correct from hospital records.
4. Missing TreatmentCost -> Replace using diagnosis median cost.
5. Invalid Age (>120) -> Replace using median valid age.
6. Missing Age -> Replace using median valid age.
7. Invalid Diagnosis -> Standardize using approved diagnosis names.
""")


print("\n" + "="*60)
print("QUESTION 3: STANDARDIZE DIAGNOSIS NAMES")
print("="*60)

df["Diagnosis"] = df["Diagnosis"].str.strip().str.title()
df["Diagnosis"] = df["Diagnosis"].replace({
    "Diabtes": "Diabetes"
})

print("Standardized Diagnoses:")
print(df["Diagnosis"].unique())


print("\n" + "="*60)
print("QUESTION 4: EFFECT OF CLEANING")
print("="*60)

df["AdmissionDate"] = pd.to_datetime(
    df["AdmissionDate"], dayfirst=True, errors="coerce"
)

df["DischargeDate"] = pd.to_datetime(
    df["DischargeDate"], dayfirst=True, errors="coerce"
)

df["StayDays_Raw"] = (
    df["DischargeDate"] - df["AdmissionDate"]
).dt.days

raw_avg_cost = df["TreatmentCost"].mean()
raw_avg_stay = df["StayDays_Raw"].mean()

df["PatientID"] = df["PatientID"].fillna("UNKNOWN_001")

df.loc[df["Age"] > 120, "Age"] = np.nan
df["Age"] = df["Age"].fillna(df["Age"].median())

df = df.drop_duplicates()

df.loc[df["DischargeDate"] < df["AdmissionDate"], "DischargeDate"] = pd.NaT

df["TreatmentCost"] = df["TreatmentCost"].fillna(
    df.groupby("Diagnosis")["TreatmentCost"].transform("median")
)

df["StayDays"] = (
    df["DischargeDate"] - df["AdmissionDate"]
).dt.days

clean_avg_cost = df["TreatmentCost"].mean()
clean_avg_stay = df["StayDays"].mean()

print("\nBefore Cleaning:")
print("Average Treatment Cost =", round(raw_avg_cost, 2))
print("Average Hospital Stay  =", round(raw_avg_stay, 2), "days")

print("\nAfter Cleaning:")
print("Average Treatment Cost =", round(clean_avg_cost, 2))
print("Average Hospital Stay  =", round(clean_avg_stay, 2), "days")

print("\nChanges:")
print("Treatment Cost Change =", round(clean_avg_cost - raw_avg_cost, 2))
print("Stay Duration Change  =", round(clean_avg_stay - raw_avg_stay, 2), "days")


print("\n" + "="*60)
print("QUESTION 5: DATA GOVERNANCE RECOMMENDATIONS")
print("="*60)

print("""
1. Use mandatory PatientID validation.
2. Prevent duplicate records using unique PatientID rules.
3. Validate admission and discharge dates automatically.
4. Use standard diagnosis codes such as ICD codes.
5. Set valid age ranges during data entry.
6. Make treatment cost fields mandatory.
7. Use a common date format across departments.
8. Perform regular data-quality audits.
9. Provide data-entry training to hospital staff.
10. Maintain role-based access and change logs.
""")


print("\n" + "="*60)
print("FINAL CLEANED DATA")
print("="*60)

print(df)


QUESTION 1: DETECT INCONSISTENCIES AND ANOMALIES

Missing Patient IDs:
  PatientID     Name   Age Gender AdmissionDate DischargeDate Diagnosis  \
3       NaN  Kavya R  29.0      F    15-01-2025    17-01-2025   Diabtes   

   TreatmentCost DoctorID  
3        15000.0     D103  

Duplicate Records:
  PatientID     Name   Age Gender AdmissionDate DischargeDate Diagnosis  \
4     P1005  Rahul P  51.0      M    18-01-2025    25-01-2025    Asthma   
5     P1005  Rahul P  51.0      M    18-01-2025    25-01-2025    Asthma   

   TreatmentCost DoctorID  
4            NaN     D104  
5            NaN     D104  

Discharge Before Admission:
  PatientID     Name   Age Gender AdmissionDate DischargeDate     Diagnosis  \
1     P1002  Priya S  38.0      F    12-01-2025    11-01-2025  Hypertension   

   TreatmentCost DoctorID  
1        18000.0     D102  

Missing Treatment Cost:
  PatientID     Name   Age Gender AdmissionDate DischargeDate Diagnosis  \
4     P1005  Rahul P  51.0      M    18-01-2025

3.

In [4]:
import pandas as pd
import numpy as np

data = {
    "CustomerID": ["L001","L002","L003","L004","L005","L006","L007","L008","L008","L010"],
    "AnnualIncome": ["750000","900K","1200000", "150000","5000000","450000","75000000","650000","650000",None],
    "CreditScore": [720,680,None,450,790,350,800,690,690,710],
    "LoanAmount": [300000,400000,600000,-200000,1000000,250000,500000,350000,350000,300000],
    "EmploymentStatus": ["Full Time","FT","fulltime","Part Time","Self Employed","Unemployed","Full Time","FT","FT","Full Time"],
    "Age": [35,42,28,30,16,40,45,33,33,37],
    "ExistingDebt": [50000,100000,200000,50000,10000,300000,100000,70000,70000,80000],
    "LoanStatus": ["Approved","Approved","Approved","Rejected","Approved","Rejected","Approved","Approved","Approved","Approved"]
}

df = pd.DataFrame(data)

print("\n" + "="*60)
print("QUESTION 1: CRITICAL ISSUES IMPACTING LOAN DECISIONS")
print("="*60)

print("\nMissing Credit Scores:")
print(df[df["CreditScore"].isnull()])

print("\nDifferent Income Formats:")
print(df["AnnualIncome"])

print("\nNegative Loan Amounts:")
print(df[df["LoanAmount"] < 0])

print("\nDuplicate Customer IDs:")
print(df[df.duplicated("CustomerID", keep=False)])

print("\nInvalid Ages:")
print(df[df["Age"] < 18])

print("\nInconsistent Employment Status:")
print(df["EmploymentStatus"].unique())

print("\nExtreme Income Values:")
print(df[df["AnnualIncome"].astype(str).str.replace("K","000", regex=False).astype(float) > 10000000])


print("\n" + "="*60)
print("QUESTION 2: CLEANING TECHNIQUES")
print("="*60)

print("""
1. Missing CreditScore -> Replace with median credit score.
2. Income units -> Convert all income values to INR.
3. Negative LoanAmount -> Treat as invalid and remove/correct.
4. Duplicate CustomerID -> Remove duplicate records.
5. Age below 18 -> Treat as invalid and verify from source.
6. EmploymentStatus -> Standardize FT, Full Time and fulltime.
7. Missing Income -> Replace using median income or verify source.
""")


print("\n" + "="*60)
print("QUESTION 3: OUTLIER DETECTION AND TREATMENT")
print("="*60)

df["Income"] = df["AnnualIncome"].str.replace(
    "K", "000", regex=False
).astype(float)

Q1 = df["Income"].quantile(0.25)
Q3 = df["Income"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df["Income"] < lower) |
    (df["Income"] > upper)
]

print("\nIncome Outliers:")
print(outliers[["CustomerID","Income"]])

print("\nOutlier Treatment:")
print("Extreme income values are checked against source records.")
print("Verified values are retained; incorrect values are capped or corrected.")


print("\n" + "="*60)
print("QUESTION 4: EFFECT OF CLEANING ON LOAN APPROVAL RATE")
print("="*60)

raw_approval_rate = (
    df["LoanStatus"].eq("Approved").mean() * 100
)

df["CustomerID"] = df["CustomerID"].astype(str)

df["CreditScore"] = df["CreditScore"].fillna(
    df["CreditScore"].median()
)

df["Income"] = df["Income"].fillna(
    df["Income"].median()
)

df["EmploymentStatus"] = df["EmploymentStatus"].replace({
    "FT": "Full Time",
    "fulltime": "Full Time"
})

df = df.drop_duplicates("CustomerID")

df = df[df["LoanAmount"] >= 0]

df = df[df["Age"] >= 18]

clean_approval_rate = (
    df["LoanStatus"].eq("Approved").mean() * 100
)

print("\nBefore Cleaning:")
print("Total Customers =", 10)
print("Approved Loans =", (data["LoanStatus"]).count("Approved"))
print("Approval Rate =", round(raw_approval_rate, 2), "%")

print("\nAfter Cleaning:")
print("Total Customers =", len(df))
print("Approved Loans =", (df["LoanStatus"] == "Approved").sum())
print("Approval Rate =", round(clean_approval_rate, 2), "%")

print("\nChange in Approval Rate:")
print(
    round(clean_approval_rate - raw_approval_rate, 2),
    "%"
)


print("\n" + "="*60)
print("QUESTION 5: ETHICAL IMPLICATIONS")
print("="*60)

print("""
1. Incorrect cleaning may unfairly reject eligible customers.
2. Removing valid outliers can discriminate against high-income customers.
3. Incorrect credit-score imputation can change loan decisions.
4. Missing data should not automatically be interpreted as poor credit.
5. Cleaning rules must be transparent and consistently applied.
6. Customer financial data must be protected and kept confidential.
7. Automated loan decisions should be monitored for bias.
8. Important decisions should allow human review when necessary.
""")


print("\n" + "="*60)
print("FINAL CLEANED DATA")
print("="*60)

print(df)


QUESTION 1: CRITICAL ISSUES IMPACTING LOAN DECISIONS

Missing Credit Scores:
  CustomerID AnnualIncome  CreditScore  LoanAmount EmploymentStatus  Age  \
2       L003      1200000          NaN      600000         fulltime   28   

   ExistingDebt LoanStatus  
2        200000   Approved  

Different Income Formats:
0      750000
1        900K
2     1200000
3      150000
4     5000000
5      450000
6    75000000
7      650000
8      650000
9         NaN
Name: AnnualIncome, dtype: str

Negative Loan Amounts:
  CustomerID AnnualIncome  CreditScore  LoanAmount EmploymentStatus  Age  \
3       L004       150000        450.0     -200000        Part Time   30   

   ExistingDebt LoanStatus  
3         50000   Rejected  

Duplicate Customer IDs:
  CustomerID AnnualIncome  CreditScore  LoanAmount EmploymentStatus  Age  \
7       L008       650000        690.0      350000               FT   33   
8       L008       650000        690.0      350000               FT   33   

   ExistingDebt LoanStat

4.

In [5]:
import pandas as pd
import numpy as np

data = {
    "SensorID": [
        "S101","S102","S103","S104","S105",
        "S106","S106","S107","S10A","S109"
    ],
    "Timestamp": [
        "01-01-2025 08:00",
        "01-01-2025 08:05",
        "01-01-2025 08:10",
        "01-01-2025 08:15",
        None,
        "01-01-2025 08:25",
        "01-01-2025 08:25",
        "01-01-2025 08:30",
        "01-01-2025 08:35",
        "01-01-2025 08:40"
    ],
    "VehicleCount": [
        120,135,-50,140,110,130,130,150,145,138
    ],
    "AverageSpeed": [
        45,52,48,350,42,46,46,55,53,50
    ],
    "TrafficSignalStatus": [
        "Green","Red","Green","Yellow","Green",
        "Red","Red","Green","Green","Green"
    ],
    "RoadName": [
        "Anna Salai","anna salai","ANNA SALAI","GST Road",
        "GST ROAD","OMR","OMR","Old Mahabalipuram Road",
        "OMR","GST Road"
    ],
    "WeatherCondition": [
        "Sunny","Sunny","Cloudy","Rainy","Sunny",
        None,None,"Sunny","Cloudy","Rainy"
    ]
}

df = pd.DataFrame(data)


print("\n" + "="*60)
print("QUESTION 1: ASSESS QUALITY OF THE TRAFFIC DATASET")
print("="*60)

print("\nMissing Timestamps:")
print(df[df["Timestamp"].isnull()])

print("\nDuplicate Sensor Readings:")
print(df[df.duplicated(keep=False)])

print("\nNegative Vehicle Counts:")
print(df[df["VehicleCount"] < 0])

print("\nInvalid Average Speeds:")
print(df[df["AverageSpeed"] > 300])

print("\nMissing Weather Information:")
print(df[df["WeatherCondition"].isnull()])

print("\nInconsistent Road Names:")
print(df["RoadName"].unique())

print("\nSensor ID Formatting Errors:")
print(df[~df["SensorID"].str.match(r"S\d{3}", na=False)])

print("\nQuality Issues:")
print("""
1. Missing timestamp       -> Completeness issue
2. Duplicate readings      -> Uniqueness issue
3. Negative vehicle count  -> Validity issue
4. Speed above 300 km/h    -> Validity issue
5. Missing weather         -> Completeness issue
6. Road name differences   -> Consistency issue
7. Invalid SensorID        -> Validity issue
""")


print("\n" + "="*60)
print("QUESTION 2: CLEANING TECHNIQUES FOR TIME-SERIES DATA")
print("="*60)

print("""
1. Convert Timestamp into datetime format.
2. Sort records according to Timestamp.
3. Remove duplicate sensor readings.
4. Replace negative vehicle counts with NaN and verify them.
5. Replace unrealistic speeds above 300 km/h with NaN.
6. Fill missing weather using nearby readings or sensor records.
7. Standardize road names.
8. Validate and correct SensorID formats.
9. Use interpolation only for suitable numerical time-series values.
""")


print("\n" + "="*60)
print("QUESTION 3: RULES TO IDENTIFY FAULTY SENSOR READINGS")
print("="*60)

print("""
1. VehicleCount < 0 -> Faulty reading.
2. AverageSpeed < 0 -> Faulty reading.
3. AverageSpeed > 300 -> Faulty reading.
4. Missing Timestamp -> Invalid reading.
5. Duplicate SensorID + Timestamp -> Duplicate reading.
6. Invalid SensorID format -> Sensor configuration error.
7. Missing WeatherCondition -> Incomplete reading.
8. Sudden extreme changes from previous readings -> Possible sensor fault.
""")


print("\n" + "="*60)
print("QUESTION 4: EFFECT OF CLEANING ON CONGESTION ANALYSIS")
print("="*60)

df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    dayfirst=True,
    errors="coerce"
)

raw_avg_speed = df["AverageSpeed"].mean()
raw_vehicle_count = df["VehicleCount"].mean()

df["RoadName"] = df["RoadName"].str.strip().str.title()

df["RoadName"] = df["RoadName"].replace({
    "Anna Salai": "Anna Salai",
    "Gst Road": "GST Road",
    "Omr": "OMR",
    "Old Mahabalipuram Road": "OMR"
})

df["VehicleCount"] = df["VehicleCount"].where(
    df["VehicleCount"] >= 0
)

df["AverageSpeed"] = df["AverageSpeed"].where(
    df["AverageSpeed"] <= 300
)

df["WeatherCondition"] = df["WeatherCondition"].fillna(
    "Unknown"
)

df["SensorID"] = df["SensorID"].replace({
    "S10A": "S108"
})

df = df.drop_duplicates()

df = df.dropna(subset=["Timestamp"])

clean_avg_speed = df["AverageSpeed"].mean()
clean_vehicle_count = df["VehicleCount"].mean()

print("\nBefore Cleaning:")
print("Average Vehicle Count =", round(raw_vehicle_count, 2))
print("Average Speed =", round(raw_avg_speed, 2), "km/h")

print("\nAfter Cleaning:")
print("Average Vehicle Count =", round(clean_vehicle_count, 2))
print("Average Speed =", round(clean_avg_speed, 2), "km/h")

print("\nChange:")
print(
    "Vehicle Count Change =",
    round(clean_vehicle_count - raw_vehicle_count, 2)
)

print(
    "Speed Change =",
    round(clean_avg_speed - raw_avg_speed, 2),
    "km/h"
)

print("""
Cleaning improves congestion analysis because invalid negative
vehicle counts and unrealistic speeds can produce false traffic
conditions. Removing duplicate readings also prevents certain
time periods from being overrepresented.
""")


print("\n" + "="*60)
print("QUESTION 5: AUTOMATED MONITORING TECHNIQUES")
print("="*60)

print("""
1. Real-time range validation for vehicle count and speed.
2. Automatic duplicate detection using SensorID and Timestamp.
3. Automatic timestamp validation.
4. Sensor ID format validation.
5. Real-time alerts for abnormal sensor readings.
6. Automated missing-value detection.
7. Dashboard for sensor health monitoring.
8. Anomaly detection using statistical or machine-learning methods.
9. Automatic data-quality reports.
10. Sensor failure alerts when readings stop arriving.
""")


print("\n" + "="*60)
print("FINAL CLEANED DATA")
print("="*60)

print(df)


QUESTION 1: ASSESS QUALITY OF THE TRAFFIC DATASET

Missing Timestamps:
  SensorID Timestamp  VehicleCount  AverageSpeed TrafficSignalStatus  \
4     S105       NaN           110            42               Green   

   RoadName WeatherCondition  
4  GST ROAD            Sunny  

Duplicate Sensor Readings:
  SensorID         Timestamp  VehicleCount  AverageSpeed TrafficSignalStatus  \
5     S106  01-01-2025 08:25           130            46                 Red   
6     S106  01-01-2025 08:25           130            46                 Red   

  RoadName WeatherCondition  
5      OMR              NaN  
6      OMR              NaN  

Negative Vehicle Counts:
  SensorID         Timestamp  VehicleCount  AverageSpeed TrafficSignalStatus  \
2     S103  01-01-2025 08:10           -50            48               Green   

     RoadName WeatherCondition  
2  ANNA SALAI           Cloudy  

Invalid Average Speeds:
  SensorID         Timestamp  VehicleCount  AverageSpeed TrafficSignalStatus  \
3   

In [1]:
import pandas as pd
import numpy as np

data = {
    "Region": [
        "Asia","Asia","Europe","Europe","North America",
        "Asia","Europe","Asia","UK","Asia"
    ],
    "Country": [
        "India","India","Germany","Germany","USA",
        "India","France","India","UK","India"
    ],
    "SalesPerson": [
        "Raj Kumar","Rajkumar","Anna M","Anna M","John D",
        "Priya S","Pierre L","Rahul P","David T","Priya S"
    ],
    "Product": [
        "Laptop","Laptop","Laptop","Laptop","Laptop",
        "Mobile","Ordinateur Portable","Mobile","Laptop","Smartphone"
    ],
    "SalesAmount": [
        50000,55000,1200,1200,1500,
        None,1300,9999999,1100,35000
    ],
    "Currency": [
        "INR","INR","EUR","EUR","USD",
        "INR","EUR","INR","GBP","INR"
    ],
    "SalesDate": [
        "10-01-2025","10-01-2025","12-01-2025","12-01-2025",
        "01-15-2025","18-01-2025","20-01-2025","25-01-2025",
        "28-01-2025","30-01-2025"
    ],
    "CustomerRating": [
        4,5,4,4,6,3,4,5,2,0
    ]
}

df = pd.DataFrame(data)


print("\n" + "="*65)
print("QUESTION 1: COMPREHENSIVE DATA-CLEANING FRAMEWORK")
print("="*65)

print("\nMissing Sales Amount:")
print(df[df["SalesAmount"].isnull()])

print("\nDuplicate Entries:")
print(df[df.duplicated(keep=False)])

print("\nInvalid Customer Ratings:")
print(df[(df["CustomerRating"] < 1) | (df["CustomerRating"] > 5)])

print("\nDifferent Product Names:")
print(df["Product"].unique())

print("\nSalesperson Variations:")
print(df["SalesPerson"].unique())

print("\nExtreme Sales Values:")
print(df[df["SalesAmount"] > 1000000])

print("""
Cleaning Framework:
1. Detect missing, duplicate and invalid records.
2. Validate numerical ranges.
3. Standardize currencies, products and dates.
4. Correct salesperson spelling variations.
5. Detect and treat extreme outliers.
6. Validate the cleaned dataset before analysis.
""")


print("\n" + "="*65)
print("QUESTION 2: STANDARDIZE CURRENCIES, PRODUCTS AND DATES")
print("="*65)

print("\nCurrency Standardization:")

exchange_rates = {
    "INR": 1,
    "USD": 85,
    "EUR": 92,
    "GBP": 108
}

df["Sales_INR"] = df["SalesAmount"] * df["Currency"].map(
    exchange_rates
)

print(df[["SalesAmount","Currency","Sales_INR"]])

df["Product"] = df["Product"].replace({
    "Ordinateur Portable": "Laptop",
    "Smartphone": "Mobile"
})

df["SalesPerson"] = df["SalesPerson"].replace({
    "Rajkumar": "Raj Kumar"
})

df["SalesDate"] = pd.to_datetime(
    df["SalesDate"],
    dayfirst=True,
    errors="coerce"
)

print("\nStandardized Products:")
print(df["Product"].unique())

print("\nStandardized Salespersons:")
print(df["SalesPerson"].unique())

print("\nStandardized Dates:")
print(df["SalesDate"])


print("\n" + "="*65)
print("QUESTION 3: MISSING VALUES AND OUTLIERS")
print("="*65)

df["Sales_INR"] = df["Sales_INR"].fillna(
    df.groupby("Product")["Sales_INR"].transform("median")
)

df["CustomerRating"] = df["CustomerRating"].where(
    df["CustomerRating"].between(1,5)
)

df["CustomerRating"] = df["CustomerRating"].fillna(
    df["CustomerRating"].median()
)

Q1 = df["Sales_INR"].quantile(0.25)
Q3 = df["Sales_INR"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df["Sales_INR"] < lower) |
    (df["Sales_INR"] > upper)
]

print("\nDetected Sales Outliers:")
print(outliers[["SalesPerson","Sales_INR"]])

df["Sales_INR"] = df["Sales_INR"].where(
    df["Sales_INR"] <= upper,
    upper
)

print("\nMissing Sales Amounts:")
print("Filled using product-wise median.")

print("\nCustomer Ratings:")
print("Invalid ratings were replaced using the median rating.")

print("\nOutliers:")
print("Extreme values were capped using the IQR upper limit.")


print("\n" + "="*65)
print("QUESTION 4: REGIONAL SALES RANKING BEFORE AND AFTER CLEANING")
print("="*65)

raw_sales = df.copy()

raw_sales["RawSales_INR"] = (
    raw_sales["SalesAmount"] *
    raw_sales["Currency"].map(exchange_rates)
)

raw_region = (
    raw_sales.groupby("Region")["RawSales_INR"]
    .sum()
    .sort_values(ascending=False)
)

df = df.drop_duplicates()

clean_region = (
    df.groupby("Region")["Sales_INR"]
    .sum()
    .sort_values(ascending=False)
)

print("\nBEFORE CLEANING:")
print(raw_region)

print("\nAFTER CLEANING:")
print(clean_region)

print("\nREGIONAL RANKING AFTER CLEANING:")

for rank, (region, sales) in enumerate(
    clean_region.items(), 1
):
    print(
        rank,
        region,
        "->",
        round(sales, 2),
        "INR"
    )

print("""
Cleaning can change regional rankings because duplicate sales,
missing values, currency differences and extreme outliers can
artificially increase or decrease regional performance.
""")


print("\n" + "="*65)
print("QUESTION 5: EXECUTIVE REPORT")
print("="*65)

print("""
EXECUTIVE REPORT
----------------

Data quality has a direct effect on strategic business decisions.

1. Revenue Accuracy
   Incorrect currencies, duplicates and outliers can produce
   incorrect total sales and regional revenue.

2. Regional Performance
   Poor-quality data can make one region appear stronger or
   weaker than it actually is.

3. Sales Forecasting
   Extreme sales values and missing values can reduce the
   accuracy of future sales forecasts.

4. Customer Analysis
   Invalid customer ratings can lead to incorrect conclusions
   about customer satisfaction.

5. Product Strategy
   Different product names can split the same product into
   multiple categories and distort product performance.

6. Employee Performance
   Salesperson spelling variations can cause sales to be
   incorrectly assigned to different employees.

7. Strategic Risk
   Management may make incorrect decisions about pricing,
   inventory, marketing and regional investment.

8. Recommendation
   The company should implement automated validation,
   standardized master data, currency conversion rules,
   duplicate detection, anomaly detection and regular
   data-quality audits.
""")


print("\n" + "="*65)
print("FINAL CLEANED DATA")
print("="*65)

print(
    df[
        [
            "Region",
            "Country",
            "SalesPerson",
            "Product",
            "Sales_INR",
            "SalesDate",
            "CustomerRating"
        ]
    ]
)


QUESTION 1: COMPREHENSIVE DATA-CLEANING FRAMEWORK

Missing Sales Amount:
  Region Country SalesPerson Product  SalesAmount Currency   SalesDate  \
5   Asia   India     Priya S  Mobile          NaN      INR  18-01-2025   

   CustomerRating  
5               3  

Duplicate Entries:
   Region  Country SalesPerson Product  SalesAmount Currency   SalesDate  \
2  Europe  Germany      Anna M  Laptop       1200.0      EUR  12-01-2025   
3  Europe  Germany      Anna M  Laptop       1200.0      EUR  12-01-2025   

   CustomerRating  
2               4  
3               4  

Invalid Customer Ratings:
          Region Country SalesPerson     Product  SalesAmount Currency  \
4  North America     USA      John D      Laptop       1500.0      USD   
9           Asia   India     Priya S  Smartphone      35000.0      INR   

    SalesDate  CustomerRating  
4  01-15-2025               6  
9  30-01-2025               0  

Different Product Names:
<StringArray>
['Laptop', 'Mobile', 'Ordinateur Portable'